# ERPs Analysis Procedure

# 1. Load raw EEG file for 1 subject

# 2. reprocess data:
  # - Filtering
  # - Artifact removal

# 3. Find event markers ("Wd2E" & "Wd2N")

# 4. Create epochs
  # - Time-locked to events|
  # - Baseline correction

# 5. Save clean epochs for 1 subject

# 6. Loop through all participants and collect clean epochs separately for "Wd2E" and "Wd2N" for each participant

# 7. Average within each subject
   # - Subject-level ERP (all E trials)
   # - Subject-level ERP (all N trials)

# 8. Average across subjects  → Grand average ERPs
   # - Grand average ERP (all E trials)
   # - Grand average ERP (all N trials)

Summary: Why We Decided Not to Use ASR

We initially planned to apply **Artifact Subspace Reconstruction (ASR)** for cleaning EEG data, due to its ability to **dynamically suppress transient, high-amplitude artifacts** without discarding large amounts of data. However, we decided to **pause ASR implementation for now** for the following reasons:

1. **Technical Limitations in Colab**:
   The primary Python-based ASR implementation (`pyPREP`) cannot be reliably installed or executed in Google Colab due to dependencies on R and other low-level libraries that are incompatible with the Colab environment.

2. **MATLAB/EEGLAB Dependency**:
   While ASR is natively available in EEGLAB (MATLAB), using it would require:

   * Opening each `.set` file in EEGLAB
   * Running ASR manually or via script
   * Saving cleaned files
   * Transferring them back to the original folder structure for further analysis in Python
     This approach is **time-consuming and introduces additional file management overhead**.

3. **MNE built-in** asrpy method


In [ ]:
# =============================================
# 0. SETUP
# =============================================
from google.colab import drive
drive.mount('/content/drive')

!pip install mne --quiet
import mne
import os
import numpy as np
from scipy.signal import hilbert
import matplotlib.pyplot as plt

In [ ]:
# =============================================
# 1. PARAMETERS
# =============================================
root_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEG Data' # main data directory
output_dir = os.path.join(root_dir, 'Group_Level_Results') #saving processed results
os.makedirs(output_dir, exist_ok=True)

event_id = {'Wd2N': 1, 'Wd2E': 2}  # Event codes “Wd2N” = slowed context, “Wd2E” = normal contex
tmin, tmax = -0.1, 0.8  # Epoch window: from -100 ms to 800 ms around event onset.
baseline = (tmin, 0)     # Baseline correction

In [ ]:
# =============================================
# 2. PROCESSING FUNCTIONS
# =============================================
def process_file(raw_file):
    """Process single EEG .set file."""
    try:
        # Load one raw EEG data
        raw = mne.io.read_raw_eeglab(raw_file, preload=True)
        raw.filter(0.1, None)      # High-pass filter above 0.1 Hz to remove slow drifts
        raw.notch_filter(60)       # Notch filter at 60 Hz to remove line noise

        # Artifact removal using Independent Component Analysis (ICA)here
        ica = mne.preprocessing.ICA(n_components=15, random_state=97, max_iter='auto')
        ica.fit(raw)
        raw = ica.apply(raw)

        # Extract events markers (“Wd2E”, “Wd2N”) from the file’s annotations
        events, event_dict = mne.events_from_annotations(raw)

        # Creates epochs around event markers useing event_id dict to find “Wd2E” & “Wd2N”
        epochs = mne.Epochs(raw, events, event_id=event_id, tmin=tmin, tmax=tmax,
                            baseline=baseline, preload=True, reject_by_annotation=True)

        # Baseline correction for DC offsets by subtracting mean of baseline period.
        epochs.apply_baseline(baseline)

        return epochs

    except Exception as e:
        print(f"Error processing {raw_file}: {str(e)}")
        return None

In [ ]:
# =============================================
# 3. MAIN PROCESSING LOOP
# =============================================
all_epochs = {'Wd2E': [], 'Wd2N': []}

participants = sorted([p for p in os.listdir(root_dir)
                       if os.path.isdir(os.path.join(root_dir, p))])

for participant in participants:
    participant_path = os.path.join(root_dir, participant)
    epoched_file = os.path.join(participant_path, f"{participant}_epochs-epo.fif")

    # Load preprocessed epochs or process .set file
    if os.path.exists(epoched_file):
        epochs = mne.read_epochs(epoched_file)
        print(f"Loaded preprocessed epochs for {participant}")

    else:
        # Find .set file
        set_files = [f for f in os.listdir(participant_path) if f.endswith('.set')]
        if not set_files:
            print(f"No .set file found for {participant}")
            continue

        # Process .set file
        raw_file = os.path.join(participant_path, set_files[0])
        epochs, envelope = process_file(raw_file)

        if epochs:
            epochs.save(epoched_file, overwrite=True)
        else:
            continue

    # Separate and store for each trail
    for cond in event_id:
        if cond in epochs.event_id:
            cond_epochs = epochs[cond]
            all_epochs[cond].append(cond_epochs)

In [ ]:
# =============================================
# 4. GROUP-LEVEL ANALYSIS
# =============================================

# Compute epochs for each subject
evokeds_E = [epochs.average() for epochs in all_epochs['Wd2E']]
evokeds_N = [epochs.average() for epochs in all_epochs['Wd2N']]

# Compute grand average ERP for each condition
erp_E = mne.grand_average(evokeds_E)
erp_N = mne.grand_average(evokeds_N)

In [ ]:
print(erp_E.info['ch_names'])

In [ ]:
# =============================================
# 5. PLOTTIN ERPs (9 REGIONS GRID)
# =============================================

# Define 9 scalp regions with anatomically relevant channels
regions = {
    'Left Anterior':     ['E9', 'E2', 'E3', 'E4', 'E5'],
    'Medial Anterior':   ['E6', 'E11', 'E12', 'E13', 'E16'],
    'Right Anterior':    ['E117', 'E118', 'E119', 'E120', 'E123'],
    'Left Central':      ['E45', 'E46', 'E41', 'E36', 'E42'],
    'Medial Central':    ['E13', 'E31', 'E80', 'E112', 'E7'],
    'Right Central':     ['E105', 'E111', 'E104', 'E108', 'E102'],
    'Left Posterior':    ['E50', 'E51', 'E52', 'E59', 'E60'],
    'Medial Posterior':  ['E61', 'E62', 'E67', 'E72', 'E77'],
    'Right Posterior':   ['E85', 'E86', 'E91', 'E97', 'E101']
}

fig, axs = plt.subplots(3, 3, figsize=(15, 10), sharex=True, sharey=True)

for i, (region, ch_list) in enumerate(regions.items()):
    ax = axs[i // 3, i % 3]

    # Keep only the channels that exist in the data and aren’t bad
    good_ch_list = [ch for ch in ch_list if ch in erp_E.ch_names and ch not in erp_E.info['bads']]
    if not good_ch_list:
        continue

    ch_indices = [erp_E.ch_names.index(ch) for ch in good_ch_list]

    # Average across channels in this region, ignoring NaNs
    erp_E_region = np.nanmean(erp_E.data[ch_indices, :], axis=0)
    erp_N_region = np.nanmean(erp_N.data[ch_indices, :], axis=0)

    # Plot ERP waveforms
    ax.plot(erp_E.times * 1000, erp_E_region * 1e6, color='black', label='Normal Context', linewidth=1.2)
    ax.plot(erp_E.times * 1000, erp_N_region * 1e6, color='gray', label='Slowed Context', linewidth=1.2)

    ax.set_title(region, fontsize=10)
    ax.axvline(0, color='k', linestyle='--', linewidth=0.5)
    ax.axhline(0, color='k', linestyle='--', linewidth=0.5)
    ax.set_xlim(-100, 400)
    ax.set_ylim(-30, 30)
    if i == 0:
        ax.legend(loc='upper left', fontsize=8, frameon=False)

# Global axis labels
axs[-1, 1].set_xlabel('Time (ms)', fontsize=12)
axs[1, 0].set_ylabel('Amplitude (μV)', fontsize=12)
plt.suptitle('ERP Waveforms in 9 Scalp Regions (Normalized to μV)', fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()



In [ ]:
import os
import mne
import matplotlib.pyplot as plt

# Directory where your EEG files are stored

eeg_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025' # main data directory


# List all files (assuming .fif format; adjust if using other formats)
eeg_files = [f for f in os.listdir(eeg_dir) if f.endswith('.fif')]

# ICA parameters
n_components = 20
random_state = 97

for eeg_file in eeg_files:
    print(f"Processing file: {eeg_file}")

    file_path = os.path.join(eeg_dir, eeg_file)

    # Load raw EEG file
    raw = mne.io.read_raw_fif(file_path, preload=True)

    # Optional: Filter before ICA if not already done
    raw.filter(l_freq=1.0, h_freq=40.0)
    raw.notch_filter(freqs=50)  # Or 60 if applicable

    # Fit ICA
    ica = mne.preprocessing.ICA(n_components=n_components, random_state=random_state, max_iter='auto')
    ica.fit(raw)

    # Plot ICA components for manual inspection
    ica.plot_components()
    plt.show()

    # Plot ICA sources
    ica.plot_sources(raw)
    plt.show()

    # MANUAL STEP: After inspecting, specify bad components
    # You can pause code here, or create a list of bad components manually
    # For example (replace with your actual selection):
    ica.exclude = [0, 1]  # <-- replace with your selected bad components

    # Apply ICA cleaning
    raw_clean = ica.apply(raw)

    # Save cleaned data
    cleaned_filename = eeg_file.replace('.fif', '_cleaned.fif')
    cleaned_path = os.path.join(eeg_dir, cleaned_filename)
    raw_clean.save(cleaned_path, overwrite=True)

    print(f"Finished ICA cleaning for: {eeg_file}\n")

print("All files processed!")


In [ ]:
import os
import mne
import numpy as np

# Your EEG directory where ICA solutions are saved
eeg_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025'
eeg_files = [f for f in os.listdir(eeg_dir) if f.endswith('_raw.fif')]

# Loop through each file and evaluate ICA
for eeg_file in eeg_files:
    subject_id = eeg_file.replace('_raw.fif', '')
    raw_path = os.path.join(eeg_dir, eeg_file)
    cleaned_path = os.path.join(eeg_dir, f'{subject_id}_cleaned.fif')

    if not os.path.exists(cleaned_path):
        print(f"Cleaned file missing for: {subject_id} -- skipping")
        continue

    print(f"\nEvaluating ICA for subject: {subject_id}")

    # Load raw data
    raw = mne.io.read_raw_fif(raw_path, preload=True)

    # Reload ICA object if you saved it (assuming you have an ICA solution saved)
    # Otherwise: you must run ICA again here to retrieve component info.
    ica_path = os.path.join(eeg_dir, f'{subject_id}_ica.fif')
    if os.path.exists(ica_path):
        ica = mne.preprocessing.read_ica(ica_path)
    else:
        # IF ICA was not saved, refit ICA (note: not ideal, better to load saved ICA!)
        ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter='auto')
        ica.fit(raw)

    # 1️⃣ Explained variance
    explained_var = ica.get_explained_variance_ratio(raw)
    total_explained = np.sum(explained_var) * 100
    print(f"Total variance explained by ICA: {total_explained:.2f}%")

    # 2️⃣ Automatic EOG detection (if EOG channels exist)
    try:
        eog_inds, eog_scores = ica.find_bads_eog(raw)
        print(f"EOG-related components automatically detected: {eog_inds}")
    except Exception as e:
        print("No EOG channels available for automatic blink detection.")

    # 3️⃣ Automatic ECG detection (if ECG channels exist)
    try:
        ecg_inds, ecg_scores = ica.find_bads_ecg(raw)
        print(f"ECG-related components automatically detected: {ecg_inds}")
    except Exception as e:
        print("No ECG channels available for heartbeat detection.")

    # 4️⃣ Before vs After standard deviation (data variance reduction)
    raw_clean = mne.io.read_raw_fif(cleaned_path, preload=True)
    before_std = raw.get_data().std()
    after_std = raw_clean.get_data().std()
    reduction = ((before_std - after_std) / before_std) * 100
    print(f"Overall variance reduction after ICA: {reduction:.2f}%")

    print("----------------------------------------------------")

print("\nAll ICA evaluations completed!")


In [ ]:
import os
import mne
import numpy as np
import matplotlib.pyplot as plt

eeg_dir = '/content/drive/My Drive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025'
eeg_files = [f for f in os.listdir(eeg_dir) if f.endswith('_raw.fif')]

for eeg_file in eeg_files:
    subject_id = eeg_file.replace('_raw.fif', '')
    raw_path = os.path.join(eeg_dir, eeg_file)
    cleaned_path = os.path.join(eeg_dir, f'{subject_id}_cleaned.fif')
    ica_path = os.path.join(eeg_dir, f'{subject_id}_ica.fif')

    if not os.path.exists(cleaned_path) or not os.path.exists(ica_path):
        print(f"Cleaned or ICA file missing for: {subject_id} -- skipping")
        continue

    print(f"\nEvaluating ICA for subject: {subject_id}")

    # Load raw and ICA
    raw = mne.io.read_raw_fif(raw_path, preload=True)
    raw_clean = mne.io.read_raw_fif(cleaned_path, preload=True)
    ica = mne.preprocessing.read_ica(ica_path)

    # Plot ICA components (block execution until you close the window)
    ica.plot_components(title=f'{subject_id} - ICA Components')
    plt.show(block=True)

    # Plot ICA sources
    ica.plot_sources(raw, title=f'{subject_id} - ICA Sources')
    plt.show(block=True)

    # Plot comparison between raw and cleaned
    mne.viz.plot_compare_raws(raw, raw_clean, title=f'{subject_id} - Before vs After ICA')
    plt.show(block=True)

    input("Press Enter to continue to next subject...")

print("\nAll ICA evaluations completed!")
